In [ ]:
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_dir = 'models/'
inference_data_path = 'data/fb2022_inference.csv.gz'
output_path = 'data/fb2022_predicted_goals_bert_all.csv.gz'

inference = pd.read_csv(inference_data_path)
inference = inference[inference['text'] != ""]
inference = inference.dropna()

goals = ['DONATE', 'CONTACT', 'PURCHASE', 'GOTV', 'EVENT', 'POLL', 'GATHERINFO', 'LEARNMORE', "PRIMARY_PERSUADE"]

# Tokenize data
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
inference_dataset = Dataset.from_pandas(inference)
def tokenize_and_pad(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)
    return tokenized
inference_dataset = inference_dataset.map(tokenize_and_pad, batched=True)

# Loop over goals
df_output = pd.DataFrame({'ad_id': inference['ad_id']})
for goal in goals:

    # Load the saved model
    model_path = model_dir + "/" + goal  # Path to the saved model directory
    model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)

    # Define batch size for inference
    batch_size = 2000  # Adjust this as needed; 2k takes about 10GB

    predicted_labels = []

    # Loop over the inference dataset in batches
    for batch_start in range(0, len(inference_dataset), batch_size):
        batch = inference_dataset[batch_start:batch_start + batch_size]

        # Convert input_ids to a list of tensors
        input_ids = [torch.tensor(ids).to(device) for ids in batch['input_ids']]

        # Pad sequences to a common length on the GPU
        input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id).to(device)

        # Create attention masks on the GPU
        attention_masks = (input_ids != tokenizer.pad_token_id).to(torch.int64)

        # Perform inference on the GPU
        with torch.no_grad():
            logits = model(input_ids, attention_mask=attention_masks).logits

        # Get predicted class labels and extend the list
        predicted_labels.extend(torch.argmax(logits, dim=1).tolist())

        print("Finished batch starting with example" + str(batch_start) + " for goal: " + goal)

    # Predicted labels are in 'predicted_labels' as a list
    df_output[goal] = predicted_labels
    df_output.to_csv(output_path, index = False)

# Rename column names to goal names used in the paper
goal_names = {
    "DONATE": "Donate",
    "CONTACT": "Contact",
    "PURCHASE": "Purchase",
    "GOTV": "Vote",
    "EVENT": "Event",
    "POLL": "Poll",
    "GATHERINFO": "Acquisition",
    "LEARNMORE": "Learn",
    "PRIMARY_PERSUADE": "Persuade"
}

fb22_pred = fb22_pred.rename(columns=goal_names)